> **Notebook d'approfondissement** — filtrage avancé : `.where()`, `.mask()`, cas limites des masques booléens, NaN dans les conditions, performance.

# Filtrer : where, mask et cas limites

In [ ]:
import sys
sys.path.append("../..")

import pandas as pd
import numpy as np
from _data import load_accidents, load_penguins

df = load_accidents()
penguins = load_penguins()

## 1. Masques booléens — rappel et cas non évidents

Un masque booléen est une Series de `True`/`False` de même longueur que le DataFrame.

In [ ]:
masque = df["dep"] == "75"
print(type(masque))
print(masque.value_counts())

In [ ]:
# Combinaisons : & (et), | (ou), ~ (non)
# OBLIGATOIRE : parenthèses autour de chaque condition
# Sans parenthèses, la précédence des opérateurs Python donne des résultats inattendus

# Correct :
df[(df["dep"] == "75") & (df["lum"] == 5)].shape

In [ ]:
# Que se passe-t-il sans parenthèses ?
try:
    df[df["dep"] == "75" & df["lum"] == 5]  # "75" & df["lum"] est évalué en premier
except Exception as e:
    print(type(e).__name__, ":", e)

### NaN dans les conditions booléennes — le comportement silencieux

Avant d'exécuter — **que retourne `df[masque_avec_nan]` quand `masque_avec_nan` contient des `NaN` ?**

In [ ]:
# penguins a des NaN dans sex — une comparaison sur sex produit des NaN dans le masque
masque_avec_nan = penguins["sex"] == "Male"
print("NaN dans le masque :", masque_avec_nan.isna().sum())

# pandas exclut silencieusement les NaN du filtre (traite NaN comme False)
print("Lignes sélectionnées :", penguins[masque_avec_nan].shape[0])
print("Dont sex == Male réels :", (penguins["sex"] == "Male").sum())

In [ ]:
# Pour inclure ou exclure explicitement les NaN
penguins[masque_avec_nan | penguins["sex"].isna()].shape  # Male + inconnus

## 2. `.isin()` — filtrer sur plusieurs valeurs

`.isin()` est plus lisible et plus rapide qu'une série de `|`.

In [ ]:
# Départements de l'Île-de-France
idf = ["75", "77", "78", "91", "92", "93", "94", "95"]

df[df["dep"].isin(idf)].shape

In [ ]:
# .isin() fonctionne aussi dans .query() via une variable @
df.query("dep in @idf").shape

In [ ]:
# Inverser : tout SAUF ces valeurs
df[~df["dep"].isin(idf)].shape

In [ ]:
%%timeit
# Performance : .isin() vs OR multiples
df[(df["dep"] == "75") | (df["dep"] == "77") | (df["dep"] == "78") |
   (df["dep"] == "91") | (df["dep"] == "92") | (df["dep"] == "93") |
   (df["dep"] == "94") | (df["dep"] == "95")]

In [ ]:
%%timeit
df[df["dep"].isin(idf)]

## 3. `.between()` — plages de valeurs

`.between(left, right)` est équivalent à `(col >= left) & (col <= right)`,
avec les deux bornes **incluses** par défaut.

In [ ]:
# Accidents en été (mois 6, 7, 8)
df[df["mois"].between(6, 8)].shape

In [ ]:
# Bornes exclusives avec inclusive="neither"
penguins[penguins["body_mass_g"].between(3500, 4500, inclusive="neither")].shape

## 4. `.where()` et `.mask()` — filtrer sans changer la forme

`.query()` et les masques booléens **réduisent** le DataFrame : le résultat a moins de lignes.

`.where()` et `.mask()` **préservent la forme** : les lignes qui ne passent pas le filtre
sont remplacées par `NaN` (ou une valeur de remplacement). Utile pour des calculs conditionnels
sans perdre l'alignement avec d'autres colonnes.

In [ ]:
# .where(condition) : garde les valeurs où condition == True, met NaN ailleurs
s = penguins["body_mass_g"]
s.where(s > 4000).head(10)  # valeurs <= 4000 deviennent NaN

In [ ]:
# .where() avec valeur de remplacement
s.where(s > 4000, other=0).head(10)  # valeurs <= 4000 remplacées par 0

In [ ]:
# .mask() est l'inverse de .where()
# .mask(condition) : met NaN où condition == True, garde les valeurs ailleurs
s.mask(s > 4000).head(10)  # valeurs > 4000 deviennent NaN

| | Résultat quand condition == True | Résultat quand condition == False |
|---|---|---|
| `.where(condition)` | Valeur originale | `NaN` (ou `other`) |
| `.mask(condition)` | `NaN` (ou `other`) | Valeur originale |

Moyen mnémotechnique : `.where()` = "garde là où c'est vrai", `.mask()` = "cache là où c'est vrai".

In [ ]:
# Cas d'usage concret : calculer une moyenne excluyant les outliers
# sans changer la forme du DataFrame
q_low  = penguins["body_mass_g"].quantile(0.05)
q_high = penguins["body_mass_g"].quantile(0.95)

penguins["body_mass_g"].where(
    penguins["body_mass_g"].between(q_low, q_high)
).mean()

## 5. `pd.cut()` et `pd.qcut()` — discrétiser avant de filtrer

Souvent utile pour créer une variable catégorielle à partir d'une variable continue,
puis filtrer ou grouper sur ces catégories.

In [ ]:
# pd.cut() : intervalles de taille fixe définis par l'utilisateur
penguins["masse_classe"] = pd.cut(
    penguins["body_mass_g"],
    bins=[2000, 3500, 4500, 7000],
    labels=["léger", "moyen", "lourd"],
)
penguins["masse_classe"].value_counts()

In [ ]:
# pd.qcut() : intervalles de même effectif (quantiles)
penguins["masse_quartile"] = pd.qcut(
    penguins["body_mass_g"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"],
)
penguins["masse_quartile"].value_counts().sort_index()

In [ ]:
# Utiliser les classes dans un groupby
penguins.groupby("masse_classe", observed=True)["bill_length_mm"].mean().round(1)

## 6. `.query()` — cas limites et moteur `numexpr`

`.query()` a quelques limitations à connaître.

In [ ]:
# Noms de colonnes qui sont des mots-clés Python ou contiennent des espaces
# → backticks
df_demo = pd.DataFrame({"class": [1, 2, 3], "ma colonne": [4, 5, 6]})
df_demo.query("`class` == 1 and `ma colonne` > 3")

In [ ]:
# .query() ne supporte pas les appels de méthode directement
# Ceci ne fonctionne pas :
try:
    df.query("dep.str.startswith('7')")
except Exception as e:
    print(type(e).__name__, ":", e)

# Solution : pré-calculer le masque dans une variable
deps_70s = df["dep"].str.startswith("7")
df[deps_70s].shape

In [ ]:
# Moteur numexpr : accélération sur les grandes expressions numériques
# Nécessite : pip install numexpr
try:
    result = df.query("mois >= 6 and mois <= 8", engine="numexpr")
    print("numexpr disponible, résultat :", result.shape)
except ImportError:
    print("numexpr non installé — moteur python utilisé par défaut")

## 7. `df.eval()` — évaluer des expressions dans le contexte du DataFrame

`.eval()` est la version "expression" de `.assign()` — elle peut être plus rapide
sur de gros volumes car elle utilise `numexpr` et évite la création d'objets intermédiaires.

In [ ]:
# Équivalent de .assign() mais en syntaxe string
df.eval("ratio_vma = vma / nbv", inplace=False)[["vma", "nbv", "ratio_vma"]].head(5)

In [ ]:
# Utile pour des expressions complexes avec plusieurs colonnes
# Ici on crée une colonne puis on filtre — tout en une passe
df.eval("""
    ratio_vma = vma / nbv
""").query("ratio_vma > 20")[["dep", "vma", "nbv", "ratio_vma"]].head(5)

## 8. Filtrage sur les index — `.loc` avec condition booléenne sur l'index

In [ ]:
# Après un set_index, l'index peut se filtrer directement
df_indexed = df.set_index("Num_Acc")

# Sélectionner des Num_Acc spécifiques
nums_a_garder = df["Num_Acc"].head(3).tolist()
df_indexed.loc[nums_a_garder, ["dep", "mois"]]

In [ ]:
# Filtrer sur une condition booléenne appliquée à l'index
# Ici : garder les Num_Acc pairs (à titre illustratif)
df_indexed[df_indexed.index % 2 == 0].head(3)

## Récapitulatif des outils de filtrage

| Besoin | Outil |
|---|---|
| Filtrer des lignes, résultat réduit | `.query()` (défaut) ou masque `df[condition]` |
| Plusieurs valeurs possibles | `.isin([...])` |
| Plage numérique | `.between(low, high)` |
| Masque mais sans changer la forme | `.where(condition)` / `.mask(condition)` |
| Discrétiser une variable continue | `pd.cut()` (intervalles fixes) / `pd.qcut()` (quantiles) |
| Filtrage texte sur noms de colonnes | `.filter(like=)` / `.filter(regex=)` |
| Expression numérique vectorisée rapide | `.eval()` + `engine="numexpr"` |
| NaN dans le masque | Toujours vérifier avec `.isna()` avant de filtrer |